# Phase 1 — Data Preparation & Temporal Split

**MSc Credit Risk Project — Colab handover (1 of 6)**

Applies the full cleaning and feature-engineering pipeline (the exact same code
that produced the dissertation results) and creates the temporal split:

- `DAYS_EMPLOYED = 365243` sentinel correction
- Income winsorisation at the 99.9th percentile (fitted on training years only — leakage control)
- Drop 30 near-duplicate columns (found in Phase 0 EDA)
- Derived features (income/credit ratios, document counts, etc.)
- Temporal split: **train = 2018-2019, test = 2020**

**Checkpoint produced:** `data/processed/engineered_train_2018_2019.parquet` +
`engineered_test_2020.parquet`. **Runtime: ~5 minutes** (skipped entirely if the
parquets already exist on Drive).


In [ ]:
# ============================================================
# SETUP: mount Google Drive and locate the project folder
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

# AUTO-DETECT the project root (folder containing src/ and run_experiment.py).
# If auto-detection fails, set PROJECT_ROOT manually, e.g.:
#   PROJECT_ROOT = Path('/content/drive/MyDrive/credit-risk-project - Copy')
PROJECT_ROOT = None
for candidate in Path('/content/drive/MyDrive').rglob('run_experiment.py'):
    if (candidate.parent / 'src').is_dir():
        PROJECT_ROOT = candidate.parent
        break
assert PROJECT_ROOT is not None, "Could not find the project folder on Drive - set PROJECT_ROOT manually"

os.chdir(PROJECT_ROOT)
import sys
sys.path.insert(0, str(PROJECT_ROOT))
# Ensure expected directories exist (log files are opened at import time)
for _d in ('logs', 'data/raw', 'data/processed', 'models',
           'reports/tables', 'reports/figures', 'outputs/eda'):
    os.makedirs(PROJECT_ROOT / _d, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print("Working directory set. All outputs are saved here (persistent on Drive).")


In [ ]:
# Install pinned dependencies (Colab pre-installs most; these ensure version parity
# with the local artefact that produced the dissertation numbers).
!pip install -q "xgboost>=3.4,<3.5" "imbalanced-learn>=0.14,<0.15" "shap>=0.52,<0.53" "pyarrow>=25"


In [ ]:
# Verify the raw dataset is present
from src.config import RAW_APPLICATION_TRAIN
assert RAW_APPLICATION_TRAIN.exists(), (
    f"Missing {RAW_APPLICATION_TRAIN.name}!\n"
    f"Upload application_train.csv (~161 MB) into the project's data/raw/ folder on Drive,\n"
    f"then re-run this notebook."
)
print(f"Raw dataset found: {RAW_APPLICATION_TRAIN} ({RAW_APPLICATION_TRAIN.stat().st_size / 1e6:.0f} MB)")


In [ ]:
# Load, clean, engineer, split (uses the SAME code as the local artefact).
# If the engineered parquets already exist on Drive, load_and_split() loads them
# directly (bit-identical, no re-processing). Otherwise it processes the raw CSV
# and saves the parquets as checkpoints.
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from run_experiment import load_and_split
from src.config import PROCESSED_DIR

df_train, df_test = load_and_split()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
if not (PROCESSED_DIR / "engineered_train_2018_2019.parquet").exists():
    df_train.to_parquet(PROCESSED_DIR / "engineered_train_2018_2019.parquet", index=False)
    df_test.to_parquet(PROCESSED_DIR / "engineered_test_2020.parquet", index=False)
    print("Engineered parquets saved to data/processed/ (persistent on Drive).")


In [ ]:
# ---- Inspect the engineered splits ----
print(f"TRAIN (2018-2019): {df_train.shape[0]:,} rows x {df_train.shape[1]} columns, "
      f"default rate {df_train['TARGET'].mean():.4%}")
print(f"TEST  (2020):      {df_test.shape[0]:,} rows x {df_test.shape[1]} columns, "
      f"default rate {df_test['TARGET'].mean():.4%}")
df_train.head()


In [ ]:
# ---- Phase 1 verification (expected values from the dissertation) ----
assert len(df_train) == 205_007, f"train rows {len(df_train)} != 205,007"
assert len(df_test) == 102_504, f"test rows {len(df_test)} != 102,504"
assert abs(df_train['TARGET'].mean() - 0.0812) < 0.001
assert abs(df_test['TARGET'].mean() - 0.0798) < 0.001
print("PHASE 1 CHECKPOINT OK: train=205,007 (8.12% default), test=102,504 (7.98% default)")
